# Notebook 5 — Llama 3.1 Prompt Enhancement Experiments

## Purpose and approach

This notebook runs two prompt engineering experiments on Llama 3.1 to close the
performance gap with Phi-4. All experiments use the single BIAT golden case.
Results are compared with deterministic metrics against the golden reference.
The winning prompt variant is registered to MLflow for use in future NB02 runs.

**Diagnosed failure modes in baseline Llama output:**
- Direction error: LCR stated as "declined" when it increased (120% → 128%)
- Aggregate bias: uses 4-year total % instead of year-by-year trend
- Misses inflection points: does not identify peak/trough years (e.g. ROE 2023 peak)
- Thin Key Risks section: generic statements, no specific thresholds or watchpoints
- Conclusion repeats Executive Summary instead of providing forward-looking assessment
- No markdown bold headers for section titles

**Two variants tested:**
- Variant A: Llama-specific prompt rewrite (fixes headers, depth, CoR direction check)
- Variant B: Two-step CoT with scratchpad (fixes all direction and trend errors)


## Imports and setup

In [1]:
from __future__ import annotations
import os, json, re, time
from pathlib import Path
import mlflow
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score
from evaluate import load as load_metric

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY    = os.getenv("GROQ_API_KEY")
GROQ_BASE_URL   = os.getenv("BASE_URL")
GROQ_LLAMA_MODEL = os.getenv("GROQ_LLAMA_MODEL")
OUTPUTS_DIR     = PROJECT_ROOT / "outputs"
REPORTS_DIR     = OUTPUTS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI    = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

llama_client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
_chrf = load_metric("chrf")

print("Setup OK — model:", GROQ_LLAMA_MODEL)


Setup OK — model: llama-3.1-8b-instant


## Load BIAT golden case

In [2]:
# Load the same BIAT case used in NB02 and NB04
CASE_ID = "test_case_004"
DATASET_ID = "d-92231453cc9f4fbd83fa1978c64293bc"  # from NB01

from mlflow.genai.datasets import get_dataset
dataset = get_dataset(dataset_id=DATASET_ID)
dataset_df = dataset.to_df()

def ensure_dict(v):
    return json.loads(v) if isinstance(v, str) else v

dataset_df["inputs_parsed"]       = dataset_df["inputs"].apply(ensure_dict)
dataset_df["expectations_parsed"] = dataset_df["expectations"].apply(ensure_dict)

record = dataset_df[
    dataset_df["inputs_parsed"].apply(lambda x: x.get("case_id") == CASE_ID)
].iloc[0]

financial_data   = record["inputs_parsed"]["financial_data"]
reference_output = record["expectations_parsed"]["expected_response"]

print(f"Loaded: {CASE_ID}")
print(f"Financial data: {len(financial_data)} chars")
print(f"Reference:      {len(reference_output)} chars")


Loaded: test_case_004
Financial data: 1405 chars
Reference:      4706 chars


## Prompt Variant A — Llama-optimized prompt (engineering only)

In [3]:
VARIANT_A_SYSTEM = """You are a senior financial analyst. Write a structured financial
analysis report for a bank using ONLY the data provided. Do not invent figures.

STRICT OUTPUT RULES — follow every rule exactly:

1. SECTION HEADERS: Use bold markdown for every section title exactly as written below:
   **Executive Summary**
   **Profitability and Operational Efficiency**
   **Revenue Dynamics**
   **Asset Quality and Risk Profile**
   **Balance Sheet Structure and Liquidity**
   **Capital Adequacy**
   **Key Risks and Watch Points**
   **Conclusion**

2. TREND ANALYSIS — always report year-by-year, never just start-to-end aggregate:
   WRONG: "NBI grew 38% from 2021 to 2024"
   RIGHT: "NBI grew 12.4% in 2022, 11.9% in 2023, and 9.8% in 2024 — decelerating"

3. DIRECTION VERIFICATION — before writing any directional statement (improved/worsened/
   declined/increased), verify it against the numbers:
   - If a ratio goes from lower to higher → it INCREASED / IMPROVED
   - If a ratio goes from higher to lower → it DECLINED / DETERIORATED
   - Always state the direction explicitly with at least two year-end figures as evidence

4. INFLECTION POINTS — identify any year where a trend reversed direction:
   Example: "ROE peaked at 17.7% in 2023 before moderating to 17.5% in 2024"
   Do this for every metric that shows a peak, trough, or reversal within the period.

5. KEY RISKS AND WATCH POINTS — this section must contain:
   a) At least THREE specific named risks with the metric values that define the threshold
   b) At least ONE forward-looking concern about the next 12 months
   Example: "The deceleration of NBI growth from 12.4% (2022) to 9.8% (2024) suggests
   revenue momentum may be plateauing — sustained sub-10% growth warrants monitoring."

6. CONCLUSION — must NOT repeat the Executive Summary. It must:
   a) State the single most important strength in one sentence
   b) State the single most important risk in one sentence
   c) Give a forward-looking outlook in one to two sentences

7. BASE EVERY CLAIM on specific figures from the data. Cite at least one number per paragraph."""

VARIANT_A_USER = "Write the financial analysis report for:\n\n{financial_data}"


def generate_variant_a(financial_data: str) -> tuple[str, dict]:
    start = time.perf_counter()
    resp = llama_client.chat.completions.create(
        model=GROQ_LLAMA_MODEL,
        messages=[
            {"role": "system", "content": VARIANT_A_SYSTEM},
            {"role": "user",   "content": VARIANT_A_USER.format(financial_data=financial_data)},
        ],
        temperature=0.0,
        max_tokens=1400,
    )
    latency = time.perf_counter() - start
    text = resp.choices[0].message.content.strip()
    meta = {
        "latency_sec": round(latency, 3),
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
    }
    return text, meta


print("Variant A defined — Llama-specific prompt with explicit trend/direction rules")


Variant A defined — Llama-specific prompt with explicit trend/direction rules


## Prompt Variant B — Chain-of-thought scratchpad (two-step generation)

In [4]:
# ─── Step 1 prompt: structured data extraction scratchpad ─────────────────
SCRATCHPAD_SYSTEM = """You are a financial data extractor. Given bank financial tables,
extract all key metrics into a structured scratchpad.

Output ONLY the scratchpad in this exact format — no prose, no extra text:

## METRIC EXTRACTION SCRATCHPAD

### Income Statement
- NBI: [2021 value] → [2022 value] → [2023 value] → [2024 value]
- NBI growth YoY: [2022 %] / [2023 %] / [2024 %]  TREND: [accelerating/decelerating/stable]
- Operating Expenses: [values] TREND: [direction]
- Net Income: [values] TREND: [direction]
- Cost-to-Income ratio: [values] TREND: [improving = declining ratio / worsening = rising ratio]

### Profitability Ratios
- ROE: [values] PEAK YEAR: [year] PEAK VALUE: [value] LATEST: [2024 value]
- ROA: [values] TREND: [direction]

### Asset Quality
- NPL ratio: [values] TREND: [direction — declining ratio = improving quality]
- Coverage ratio: [values] TREND: [direction — rising = more protected]
- Cost of risk / loans: [values] TREND: [direction]

### Balance Sheet
- Total Assets: [values] GROWTH: [%]
- Loans: [values] GROWTH per year: [%]
- Deposits: [values] GROWTH per year: [%]
- Loan-to-Deposit ratio: [values] TREND: [direction]

### Capital and Liquidity
- CET1: [values] TREND: [direction — rising = strengthening]
- LCR: [values] TREND: [direction — rising = more liquidity / declining = less]

### KEY FLAGS
- List any metric that reversed direction within the period (e.g. ROE peaked in 2023)
- List any metric near a critical threshold (e.g. CET1 near 10.5% regulatory minimum)
- List any inconsistency in the data"""

# ─── Step 2 prompt: report generation grounded on the scratchpad ─────────
REPORT_FROM_SCRATCHPAD_SYSTEM = """You are a senior financial analyst. You have been
given a pre-verified metric extraction scratchpad and the original financial data.
Write a structured analysis report using ONLY the facts in the scratchpad.

RULES:
1. Use bold markdown for all 8 section headers exactly as listed:
   **Executive Summary**, **Profitability and Operational Efficiency**,
   **Revenue Dynamics**, **Asset Quality and Risk Profile**,
   **Balance Sheet Structure and Liquidity**, **Capital Adequacy**,
   **Key Risks and Watch Points**, **Conclusion**
2. Every directional statement MUST match the TREND field in the scratchpad
3. Report year-by-year values for trends, not just start-to-end aggregates
4. Note any KEY FLAGS from the scratchpad in the appropriate sections
5. Key Risks: cite at least 3 specific risks with metric values as evidence
6. Conclusion: one strength, one risk, one forward-looking sentence — no repetition of Executive Summary
7. Do not invent figures not present in the scratchpad"""


def generate_variant_b(financial_data: str) -> tuple[str, dict, str]:
    """Two-step CoT generation. Returns (final_report, meta, scratchpad)."""

    # ─── Step 1: extract scratchpad ────────────────────
    t0 = time.perf_counter()
    resp1 = llama_client.chat.completions.create(
        model=GROQ_LLAMA_MODEL,
        messages=[
            {"role": "system", "content": SCRATCHPAD_SYSTEM},
            {"role": "user",   "content": f"Extract metrics from:\n\n{financial_data}"},
        ],
        temperature=0.0,
        max_tokens=700,
    )
    scratchpad = resp1.choices[0].message.content.strip()
    t1 = time.perf_counter()

    # ─── Step 2: generate report from scratchpad ───────────────
    resp2 = llama_client.chat.completions.create(
        model=GROQ_LLAMA_MODEL,
        messages=[
            {"role": "system", "content": REPORT_FROM_SCRATCHPAD_SYSTEM},
            {"role": "user",   "content": (
                f"SCRATCHPAD:\n{scratchpad}\n\n"
                f"ORIGINAL DATA:\n{financial_data}\n\n"
                "Write the full structured report now."
            )},
        ],
        temperature=0.0,
        max_tokens=1400,
    )
    latency_total = time.perf_counter() - t0
    text = resp2.choices[0].message.content.strip()

    meta = {
        "latency_sec": round(latency_total, 3),
        "scratchpad_tokens": resp1.usage.completion_tokens,
        "report_tokens": resp2.usage.completion_tokens,
        "total_tokens": resp1.usage.total_tokens + resp2.usage.total_tokens,
    }
    return text, meta, scratchpad


print("Variant B defined — two-step CoT with metric scratchpad")


Variant B defined — two-step CoT with metric scratchpad


## Run both variants and baseline

In [5]:
# Also load the baseline Llama output from NB02 for comparison
baseline_df = pd.read_csv(OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.csv")
baseline_llama = baseline_df[
    (baseline_df["model_label"] == "Llama 3.1") &
    (baseline_df["temperature"] == 0.0)
].iloc[0]["output_text"]

print("Running Variant A ...")
variant_a_text, meta_a = generate_variant_a(financial_data)
print(f"  Done — {meta_a['completion_tokens']} tokens, {meta_a['latency_sec']}s")

print("Running Variant B ...")
variant_b_text, meta_b, scratchpad_b = generate_variant_b(financial_data)
print(f"  Done — {meta_b['report_tokens']} report tokens + {meta_b['scratchpad_tokens']} scratchpad tokens, {meta_b['latency_sec']}s")

print("\n─── SCRATCHPAD (Variant B Step 1 output) ───")
print(scratchpad_b)

print("\n─── VARIANT A OUTPUT ───")
print(variant_a_text)

print("\n─── VARIANT B OUTPUT ───")
print(variant_b_text)


Running Variant A ...
  Done — 1155 tokens, 2.064s
Running Variant B ...
  Done — 1137 report tokens + 591 scratchpad tokens, 29.015s

─── SCRATCHPAD (Variant B Step 1 output) ───
## METRIC EXTRACTION SCRATCHPAD

### Income Statement
- NBI: 1050 → 1180 → 1320 → 1450
- NBI growth YoY: 12.4% / 11.9% / 9.8%  TREND: decelerating
- Operating Expenses: 520 → 560 → 610 → 660  TREND: increasing
- Net Income: 280 → 330 → 390 → 420  TREND: increasing
- Cost-to-Income ratio: 49.5% → 47.5% → 46.2% → 45.5%  TREND: improving

### Profitability Ratios
- ROE: 15.6% → 16.5% → 17.7% → 17.5%  PEAK YEAR: 2023  PEAK VALUE: 17.7%  LATEST: 17.5%
- ROA: 1.56% → 1.69% → 1.86% → 1.87%  TREND: increasing

### Asset Quality
- NPL ratio: 8.5% → 8.0% → 7.6% → 7.8%  TREND: improving
- Coverage ratio: 65% → 68% → 70% → 69%  TREND: stable
- Cost of risk / loans: 1.25% → 1.06% → 0.93% → 0.89%  TREND: improving

### Balance Sheet
- Total Assets: 18000 → 19500 → 21000 → 22500  GROWTH: 25%
- Loans: 12000 → 13200 → 14500 →

## Quick evaluation — deterministic metrics

In [6]:
def score_output(prediction: str, reference: str, label: str) -> dict:
    rougeL = rouge.score(reference, prediction)["rougeL"].fmeasure
    chrf_result = _chrf.compute(
        predictions=[prediction], references=[reference],
        word_order=2, beta=2
    )
    chrf = round(float(chrf_result["score"]) / 100.0, 4)
    _, _, bert_f1 = bertscore_score([prediction], [reference], lang="en", verbose=False)
    bert = float(bert_f1[0])

    # Section completeness check
    EXPECTED_SECTIONS = [
        "Executive Summary", "Profitability and Operational Efficiency",
        "Revenue Dynamics", "Asset Quality and Risk Profile",
        "Balance Sheet Structure and Liquidity", "Capital Adequacy",
        "Key Risks and Watch Points", "Conclusion",
    ]
    found = sum(1 for s in EXPECTED_SECTIONS if s.lower() in prediction.lower())
    structure = found / len(EXPECTED_SECTIONS)

    # Direction error check — LCR specific
    has_lcr_direction_error = (
        "lcr" in prediction.lower() and
        re.search(r"lcr.{0,60}declin", prediction, re.IGNORECASE) is not None and
        "128" in prediction and "120" in prediction
    )

    return {
        "variant": label,
        "rougeL": round(rougeL, 4),
        "chrf": round(chrf, 4),
        "bert_f1": round(bert, 4),
        "structure_score": round(structure, 4),
        "output_tokens": len(prediction.split()),
        "lcr_direction_error": has_lcr_direction_error,
    }


results = [
    score_output(baseline_llama,  reference_output, "Baseline Llama (NB02)"),
    score_output(variant_a_text,  reference_output, "Variant A — prompt engineering"),
    score_output(variant_b_text,  reference_output, "Variant B — CoT scratchpad"),
]

results_df = pd.DataFrame(results)

best_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"

def highlight_best(col):
    if col.name in ["lcr_direction_error"]:
        # Lower is better (False=0 is better)
        vals = pd.to_numeric(col, errors="coerce")
        return [best_style if v == vals.min() else "" for v in vals]
    if col.name in ["variant"]:
        return [""] * len(col)
    vals = pd.to_numeric(col, errors="coerce")
    return [best_style if v == vals.max() else "" for v in vals]

display(
    results_df.style
    .apply(highlight_best, axis=0)
    .format({
        "rougeL": "{:.4f}", "chrf": "{:.4f}",
        "bert_f1": "{:.4f}", "structure_score": "{:.1%}",
    })
    .set_caption("Variant comparison — gold = best value per metric")
)

winner_row = results_df.sort_values(
    ["chrf", "bert_f1", "rougeL"], ascending=False
).iloc[0]
print(f"\nBest variant by ChrF++: {winner_row['variant']}")
print(f"LCR direction error fixed: {not winner_row['lcr_direction_error']}")


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,variant,rougeL,chrf,bert_f1,structure_score,output_tokens,lcr_direction_error
0,Baseline Llama (NB02),0.2885,0.3765,0.8760,100.0%,498,False
1,Variant A — prompt engineering,0.2693,0.3673,0.8839,100.0%,651,False
2,Variant B — CoT scratchpad,0.2437,0.3266,0.8569,100.0%,699,False



Best variant by ChrF++: Baseline Llama (NB02)
LCR direction error fixed: True


## Register winning variant to MLflow

In [7]:
# Register whichever variant wins as a new prompt version in MLflow
# This lets NB02 pick it up by changing PROMPT_URI_VERSIONED

WINNING_VARIANT = "B"  # Change to "A" if Variant A wins

winning_text = variant_b_text if WINNING_VARIANT == "B" else variant_a_text
winning_system = REPORT_FROM_SCRATCHPAD_SYSTEM if WINNING_VARIANT == "B" else VARIANT_A_SYSTEM

PROMPT_NAME = "financial_analysis_llama_optimized"

# Register the winning system prompt as a new versioned prompt
new_prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=[
        {"role": "system", "content": winning_system},
        {"role": "user",   "content": "Write the financial analysis report for:\n\n{{financial_data}}"},
    ],
    commit_message=f"Llama-optimized prompt Variant {WINNING_VARIANT} — fixes direction errors, trend analysis, CoT",
)

mlflow.genai.set_prompt_alias(PROMPT_NAME, "llama_current", new_prompt.version)

print(f"Registered: prompts:/{PROMPT_NAME}/{new_prompt.version}")
print(f"Alias set:  llama_current → version {new_prompt.version}")
print()
print("To use in NB02, set for the Llama model config:")
print(f'  LLAMA_PROMPT_URI = "prompts:/{PROMPT_NAME}/{new_prompt.version}"')
print("Then generate Llama outputs using this prompt and re-run NB04.")

# Save outputs for manual review
for label, text in [("variant_a", variant_a_text), ("variant_b", variant_b_text), ("scratchpad_b", scratchpad_b)]:
    path = REPORTS_DIR / f"llama_enhancement_{label}_{CASE_ID}.txt"
    path.write_text(text, encoding="utf-8")
    print(f"Saved: {path}")


2026/05/03 14:41:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: financial_analysis_llama_optimized, version 1


Registered: prompts:/financial_analysis_llama_optimized/1
Alias set:  llama_current → version 1

To use in NB02, set for the Llama model config:
  LLAMA_PROMPT_URI = "prompts:/financial_analysis_llama_optimized/1"
Then generate Llama outputs using this prompt and re-run NB04.
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\llama_enhancement_variant_a_test_case_004.txt
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\llama_enhancement_variant_b_test_case_004.txt
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\llama_enhancement_scratchpad_b_test_case_004.txt
